In [1]:
# Setup and listings columns
import duckdb

LST = "../data/bronze/nyc/listings.parquet"
CAL = "../data/bronze/nyc/calendar.parquet"
REV = "../data/bronze/nyc/reviews.parquet"

# List every column name in listings
cols = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{LST}')").df()
print("Number of columns:", len(cols))
print(cols["column_name"].tolist())

Number of columns: 90
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_qu

In [2]:
# Minimum and maximum nights. Are there placeholder values?
duckdb.sql(f"""
    SELECT
        'listings' AS source,
        MAX(TRY_CAST(minimum_nights AS BIGINT))                        AS max_min_nights,
        MAX(TRY_CAST(maximum_nights AS BIGINT))                        AS max_max_nights,
        COUNT(*) FILTER (WHERE TRY_CAST(maximum_nights AS BIGINT) = 2147483647)
                                                                       AS placeholder_rows,
        COUNT(*) FILTER (WHERE TRY_CAST(maximum_nights AS BIGINT) > 1125)
                                                                       AS max_nights_over_1125
    FROM read_parquet('{LST}')

    UNION ALL

    SELECT
        'calendar',
        MAX(TRY_CAST(minimum_nights AS BIGINT)),
        MAX(TRY_CAST(maximum_nights AS BIGINT)),
        COUNT(*) FILTER (WHERE TRY_CAST(maximum_nights AS BIGINT) = 2147483647),
        COUNT(*) FILTER (WHERE TRY_CAST(maximum_nights AS BIGINT) > 1125)
    FROM read_parquet('{CAL}')
""").show()

┌──────────┬────────────────┬────────────────┬──────────────────┬──────────────────────┐
│  source  │ max_min_nights │ max_max_nights │ placeholder_rows │ max_nights_over_1125 │
│ varchar  │     int64      │     int64      │      int64       │        int64         │
├──────────┼────────────────┼────────────────┼──────────────────┼──────────────────────┤
│ listings │           1124 │     2147483647 │               26 │                   33 │
│ calendar │           1124 │     2147483647 │             5210 │                 7765 │
└──────────┴────────────────┴────────────────┴──────────────────┴──────────────────────┘



In [3]:
# Review scores of zero
duckdb.sql(f"""
    SELECT
        COUNT(review_scores_rating)                                              AS has_rating,
        COUNT(*) FILTER (WHERE TRY_CAST(review_scores_rating AS DOUBLE) = 0)     AS rating_is_zero,
        MIN(TRY_CAST(review_scores_rating AS DOUBLE))                            AS min_rating,
        MAX(TRY_CAST(review_scores_rating AS DOUBLE))                            AS max_rating
    FROM read_parquet('{LST}')
""").show()

┌────────────┬────────────────┬────────────┬────────────┐
│ has_rating │ rating_is_zero │ min_rating │ max_rating │
│   int64    │     int64      │   double   │   double   │
├────────────┼────────────────┼────────────┼────────────┤
│      21700 │              1 │        0.0 │        5.0 │
└────────────┴────────────────┴────────────┴────────────┘



In [4]:
# Do calendar start dates match last_scraped?
duckdb.sql(f"""
    WITH calendar_start AS (
        SELECT listing_id, MIN(date) AS first_calendar_date
        FROM read_parquet('{CAL}')
        GROUP BY listing_id
    )
    SELECT
        l.last_scraped,
        c.first_calendar_date,
        COUNT(*) AS number_of_listings
    FROM read_parquet('{LST}') l
    JOIN calendar_start c ON l.id = c.listing_id
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show()

┌──────────────┬─────────────────────┬────────────────────┐
│ last_scraped │ first_calendar_date │ number_of_listings │
│   varchar    │       varchar       │       int64        │
├──────────────┼─────────────────────┼────────────────────┤
│ 2026-06-14   │ 2026-06-14          │              14356 │
│ 2026-06-15   │ 2026-06-14          │                  5 │
│ 2026-06-15   │ 2026-06-15          │               7123 │
│ 2026-06-22   │ 2026-06-22          │               2088 │
│ 2026-06-23   │ 2026-06-23          │               6687 │
└──────────────┴─────────────────────┴────────────────────┘



In [5]:
# Reviews: duplicates, unmatched listings, and date range
duckdb.sql(f"""
    SELECT
        COUNT(*)                                        AS total_reviews,
        COUNT(DISTINCT id)                              AS distinct_review_ids,
        MIN(date)                                       AS first_review_date,
        MAX(date)                                       AS last_review_date,
        (SELECT COUNT(DISTINCT r.listing_id)
         FROM read_parquet('{REV}') r
         LEFT JOIN read_parquet('{LST}') l ON r.listing_id = l.id
         WHERE l.id IS NULL)                            AS review_listings_not_in_listings
    FROM read_parquet('{REV}')
""").show()

┌───────────────┬─────────────────────┬───────────────────┬──────────────────┬─────────────────────────────────┐
│ total_reviews │ distinct_review_ids │ first_review_date │ last_review_date │ review_listings_not_in_listings │
│     int64     │        int64        │      varchar      │     varchar      │              int64              │
├───────────────┼─────────────────────┼───────────────────┼──────────────────┼─────────────────────────────────┤
│        990170 │              990170 │ 2009-05-25        │ 2026-06-22       │                             239 │
└───────────────┴─────────────────────┴───────────────────┴──────────────────┴─────────────────────────────────┘



In [6]:
# How often are the quote columns filled?
duckdb.sql(f"""
    SELECT
        COUNT(*)                                   AS total_listings,
        COUNT(price)                               AS has_price,
        COUNT(price_quote_price_per_night)         AS has_quote_per_night,
        COUNT(price_quote_total_price)             AS has_quote_total,
        COUNT(*) FILTER (WHERE price IS NULL
                         AND price_quote_price_per_night IS NOT NULL)
                                                   AS quote_but_no_price,
        COUNT(*) FILTER (WHERE price IS NOT NULL
                         AND price_quote_price_per_night IS NULL)
                                                   AS price_but_no_quote
    FROM read_parquet('{LST}')
""").show()

┌────────────────┬───────────┬─────────────────────┬─────────────────┬────────────────────┬────────────────────┐
│ total_listings │ has_price │ has_quote_per_night │ has_quote_total │ quote_but_no_price │ price_but_no_quote │
│     int64      │   int64   │        int64        │      int64      │       int64        │       int64        │
├────────────────┼───────────┼─────────────────────┼─────────────────┼────────────────────┼────────────────────┤
│          30259 │     21515 │               21514 │           21514 │                  0 │                  1 │
└────────────────┴───────────┴─────────────────────┴─────────────────┴────────────────────┴────────────────────┘



In [7]:
# What do the values look like?
duckdb.sql(f"""
    SELECT
        price,
        price_quote_price_per_night,
        price_quote_total_price,
        price_quote_checkin_date,
        price_quote_checkout_date,
        LEFT(price_quote_raw, 150) AS raw_preview
    FROM read_parquet('{LST}')
    WHERE price_quote_price_per_night IS NOT NULL
    LIMIT 5
""").df()

,price,price_quote_price_per_night,price_quote_total_price,price_quote_checkin_date,price_quote_checkout_date,raw_preview
0,$113.97,113.97,3419.19,2026-07-11,2026-08-10,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""..."
1,$117.27,117.27,3518.00,2026-12-20,2027-01-19,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""..."
2,$80.06,80.06,2401.84,2026-09-01,2026-10-01,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""..."
3,$77.17,77.17,2315.00,2026-12-06,2027-01-05,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""..."
4,$202.47,202.47,6074.17,2027-02-28,2027-03-30,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""..."


In [8]:
# Which dates are quoted, and does the quote match the price?
duckdb.sql(f"""
    WITH q AS (
        SELECT
            TRY_CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DOUBLE)                        AS price_num,
            TRY_CAST(REPLACE(REPLACE(price_quote_price_per_night, '$', ''), ',', '') AS DOUBLE)  AS quote_per_night,
            TRY_CAST(price_quote_checkin_date  AS DATE)                                          AS checkin,
            TRY_CAST(price_quote_checkout_date AS DATE)                                          AS checkout
        FROM read_parquet('{LST}')
        WHERE price_quote_price_per_night IS NOT NULL
    )
    SELECT
        MIN(checkin)                                         AS earliest_checkin,
        MAX(checkin)                                         AS latest_checkin,
        MEDIAN(checkout - checkin)                           AS median_nights_quoted,
        COUNT(*) FILTER (WHERE price_num IS NOT NULL)        AS both_filled,
        COUNT(*) FILTER (WHERE ROUND(price_num, 2) = ROUND(quote_per_night, 2))
                                                             AS quote_equals_price,
        MEDIAN(quote_per_night)                              AS median_quote_per_night,
        MEDIAN(price_num)                                    AS median_price
    FROM q
""").show()

┌──────────────────┬────────────────┬──────────────────────┬─────────────┬────────────────────┬────────────────────────┬──────────────┐
│ earliest_checkin │ latest_checkin │ median_nights_quoted │ both_filled │ quote_equals_price │ median_quote_per_night │ median_price │
│       date       │      date      │        double        │    int64    │       int64        │         double         │    double    │
├──────────────────┼────────────────┼──────────────────────┼─────────────┼────────────────────┼────────────────────────┼──────────────┤
│ 2026-06-14       │ 2027-06-22     │                 30.0 │       21514 │              21514 │                 174.68 │       174.68 │
└──────────────────┴────────────────┴──────────────────────┴─────────────┴────────────────────┴────────────────────────┴──────────────┘



In [9]:
# Is every quote exactly 30 nights?
duckdb.sql(f"""
    SELECT
        TRY_CAST(price_quote_checkout_date AS DATE)
          - TRY_CAST(price_quote_checkin_date AS DATE)   AS nights_quoted,
        COUNT(*)                                          AS number_of_listings
    FROM read_parquet('{LST}')
    WHERE price_quote_price_per_night IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").show()

┌───────────────┬────────────────────┐
│ nights_quoted │ number_of_listings │
│     int64     │       int64        │
├───────────────┼────────────────────┤
│            30 │              15835 │
│             1 │               2970 │
│             2 │               1038 │
│             3 │                579 │
│            31 │                279 │
│             4 │                162 │
│             5 │                158 │
│            90 │                121 │
│            60 │                 86 │
│             7 │                 55 │
└───────────────┴────────────────────┘
  10 rows                  2 columns



In [10]:
# Does total ÷ nights always equal the per-night price, and do the quoted nights relate to the minimum stay?
duckdb.sql(f"""
    WITH q AS (
        SELECT
            TRY_CAST(price_quote_total_price     AS DOUBLE)   AS total,
            TRY_CAST(price_quote_price_per_night AS DOUBLE)   AS per_night,
            TRY_CAST(minimum_nights              AS BIGINT)   AS min_nights,
            TRY_CAST(price_quote_checkout_date AS DATE)
              - TRY_CAST(price_quote_checkin_date AS DATE)    AS nights
        FROM read_parquet('{LST}')
        WHERE price_quote_price_per_night IS NOT NULL
    )
    SELECT
        COUNT(*)                                                           AS quotes,
        COUNT(*) FILTER (WHERE ABS(total / nights - per_night) <= 0.01)    AS total_matches_per_night,
        COUNT(*) FILTER (WHERE nights = min_nights)                        AS nights_equal_min_stay,
        COUNT(*) FILTER (WHERE nights > min_nights)                        AS nights_above_min_stay,
        COUNT(*) FILTER (WHERE nights < min_nights)                        AS nights_below_min_stay
    FROM q
""").show()

┌────────┬─────────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┐
│ quotes │ total_matches_per_night │ nights_equal_min_stay │ nights_above_min_stay │ nights_below_min_stay │
│ int64  │          int64          │         int64         │         int64         │         int64         │
├────────┼─────────────────────────┼───────────────────────┼───────────────────────┼───────────────────────┤
│  21514 │                   21514 │                 20748 │                   765 │                     0 │
└────────┴─────────────────────────┴───────────────────────┴───────────────────────┴───────────────────────┘



In [11]:
# What's inside price_quote_raw?
import json

raw = duckdb.sql(f"""
    SELECT price_quote_raw
    FROM read_parquet('{LST}')
    WHERE price_quote_raw IS NOT NULL
    LIMIT 1
""").fetchone()[0]

# Convert the JSON text into a Python dictionary and print it neatly
print(json.dumps(json.loads(raw), indent=2))

{
  "quote": {
    "taxes": null,
    "currency": "USD",
    "date_match": null,
    "service_fee": null,
    "total_price": "3419.19",
    "cleaning_fee": null,
    "is_available": true,
    "discount_amount": "6979.00",
    "price_per_night": "113.973",
    "nightly_subtotal": null,
    "discounted_subtotal": "3419.19",
    "raw_price_line_items": [
      {
        "amount": "10398.19",
        "item_type": "other",
        "description": "Average monthly price",
        "price_string": "$10,398.19"
      },
      {
        "amount": "-6979.00",
        "item_type": "discount_amount",
        "description": "Monthly stay discount",
        "price_string": "-$6,979.00"
      },
      {
        "amount": "3419.19",
        "item_type": "discounted_subtotal",
        "description": "Price after discount",
        "price_string": "$3,419.19"
      }
    ],
    "returned_checkin_date": null,
    "requested_checkin_date": "2026-07-11",
    "returned_checkout_date": null,
    "requested_che

In [12]:
# how common are discounts and fees?
duckdb.sql(f"""
    WITH j AS (
        SELECT
            TRY_CAST(price_quote_checkout_date AS DATE)
              - TRY_CAST(price_quote_checkin_date AS DATE)                                  AS nights,
            TRY_CAST(price_quote_total_price AS DOUBLE)                                     AS total,
            TRY_CAST(json_extract_string(price_quote_raw, '$.quote.discount_amount') AS DOUBLE)
                                                                                            AS discount,
            json_extract_string(price_quote_raw, '$.quote.cleaning_fee')                    AS cleaning_fee,
            json_extract_string(price_quote_raw, '$.quote.service_fee')                     AS service_fee,
            json_extract_string(price_quote_raw, '$.quote.taxes')                           AS taxes,
            json_extract_string(price_quote_raw, '$.quote.currency')                        AS currency
        FROM read_parquet('{LST}')
        WHERE price_quote_raw IS NOT NULL
    )
    SELECT
        CASE
            WHEN nights < 7  THEN '1-6 nights'
            WHEN nights < 28 THEN '7-27 nights'
            ELSE '28+ nights'
        END                                                        AS stay_length,
        COUNT(*)                                                   AS quotes,
        COUNT(*) FILTER (WHERE discount > 0)                       AS has_discount,
        ROUND(MEDIAN(100 * discount / (total + discount))
              FILTER (WHERE discount > 0), 1)                      AS median_discount_pct,
        COUNT(cleaning_fee)                                        AS has_cleaning_fee,
        COUNT(service_fee)                                         AS has_service_fee,
        COUNT(taxes)                                               AS has_taxes,
        COUNT(DISTINCT currency)                                   AS number_of_currencies
    FROM j
    GROUP BY 1
    ORDER BY MIN(nights)
""").show()

┌─────────────┬────────┬──────────────┬─────────────────────┬──────────────────┬─────────────────┬───────────┬──────────────────────┐
│ stay_length │ quotes │ has_discount │ median_discount_pct │ has_cleaning_fee │ has_service_fee │ has_taxes │ number_of_currencies │
│   varchar   │ int64  │    int64     │       double        │      int64       │      int64      │   int64   │        int64         │
├─────────────┼────────┼──────────────┼─────────────────────┼──────────────────┼─────────────────┼───────────┼──────────────────────┤
│ 1-6 nights  │   4987 │          339 │                18.6 │                0 │               0 │         8 │                    1 │
│ 7-27 nights │     83 │           28 │                 4.1 │                0 │               0 │         0 │                    1 │
│ 28+ nights  │  17608 │         8751 │                17.6 │                0 │               0 │        50 │                    1 │
└─────────────┴────────┴──────────────┴─────────────────────┴─